# Portfolio Analysis — Assignment 1 (Challenge Version)

Mean-variance analysis of a $100 million fund invested in **six U.S.-listed common stocks** selected from WRDS (CRSP monthly returns, January 2020 – December 2024), with the **one-month U.S. Treasury bill** as the risk-free asset.

**Method.** Every statistic below is computed from monthly observations and then annualized: expected returns are annualized with $\times 12$ and standard deviations with $\times \sqrt{12}$, so all numbers are directly comparable. Sharpe ratios are computed on annualized numbers using the annualized average T-bill rate. Optimization uses `scipy.optimize.minimize` (SLSQP), which handles the nonlinear variance objective and equality constraints directly.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
STOCK_FILE = DATA_DIR / "crsp_monthly.csv"
RF_FILE = DATA_DIR / "rf_monthly.csv"

OUTPUT_DIR.mkdir(exist_ok=True)

# Long-only portfolio by default; set to True to allow short sales
ALLOW_SHORT = False
MONTHS_PER_YEAR = 12

## Task 1 — Data downloaded from WRDS

**Asset selection.** Six U.S.-listed common stocks covering more than three industries, each with a complete 60-month return history (Jan 2020 – Dec 2024):

| Ticker | Industry |
|--------|----------|
| MSFT | Technology | 
| NVDA | Technology (semiconductors) |
| JPM | Financials |
| XOM | Energy |
| WMT | Consumer staples |
| *(6th asset)* | *(add your WRDS export to `data/crsp_monthly.csv` and update this table)* |

**Source.** WRDS → CRSP Monthly Stock File (fields `MthCalDt`, `MthRet`, `Ticker`), exported to `data/crsp_monthly.csv`.

**Risk-free rate.** One-month U.S. Treasury bill rate for the same period, downloaded from WRDS and saved to `data/rf_monthly.csv` with columns `MthCalDt`, `Rf` (decimal monthly rate). Source: WRDS.

The cell below loads both files, removes duplicated rows (CRSP exports occasionally contain them), pivots to a wide returns table (months × tickers), and inner-joins the T-bill series so every month has all assets plus the risk-free rate.

In [ ]:
def load_stock_data(path):
  stocks = pd.read_csv(path, parse_dates=["MthCalDt"])
  # CRSP exports occasionally contain duplicated rows; keep the first occurrence
  stocks = stocks.drop_duplicates(subset=["PERMNO", "MthCalDt"], keep="first")
  return stocks.pivot(index="MthCalDt", columns="Ticker", values="MthRet").sort_index()

def load_rf_data(path):
  if not path.exists():
    raise FileNotFoundError(
      "data/rf_monthly.csv is missing. Download the one-month T-bill rate for "
      "2020-01 to 2024-12 from WRDS and save it with columns: MthCalDt, Rf."
    )
  rf = pd.read_csv(path, parse_dates=["MthCalDt"])
  return rf.set_index("MthCalDt").sort_index()["Rf"]

stock_returns = load_stock_data(STOCK_FILE)
rf = load_rf_data(RF_FILE)

data = stock_returns.join(rf.rename("Rf"), how="inner")

tickers = [t for t in data.columns if t != "Rf"]
R = data[tickers]
n = len(tickers)

print(f"Assets: {tickers}")
print(f"Sample: {data.index.min():%Y-%m} to {data.index.max():%Y-%m} ({len(data)} months)")

assert n == 6, "The Challenge version requires exactly six risky assets"
assert len(data) == 60, "The sample must cover exactly the 60 months Jan 2020 - Dec 2024"
assert R.notna().all().all(), "Missing monthly returns - every asset needs 60 observations"

## Task 2 — Performance statistics of the six assets

For each asset, over the full 60-month sample:
- **Annualized mean return** = monthly mean × 12
- **Annualized standard deviation** = monthly sample standard deviation × $\sqrt{12}$
- **Sharpe ratio** = (annualized mean − annualized risk-free rate) / annualized standard deviation, using the mean of the monthly T-bill rates, annualized the same way.

In [ ]:
ann_mean = R.mean() * MONTHS_PER_YEAR
ann_vol = R.std() * np.sqrt(MONTHS_PER_YEAR)
ann_rf = rf.mean() * MONTHS_PER_YEAR

asset_stats = pd.DataFrame({
  "Annualized mean return": ann_mean,
  "Annualized std. deviation": ann_vol,
  "Sharpe ratio": (ann_mean - ann_rf) / ann_vol,
})

asset_stats.round(4)

## Task 3 — The equally weighted portfolio (current investment policy)

The policy allocates the $100 million **equally across the six risky assets** (1/6 each). The portfolio's monthly return is the weighted sum of the asset returns, so its expected return and variance follow directly from the monthly means and the covariance matrix. The portfolio mean over the sample is computed both ways below (statistically from mean/covariance, and empirically from realized returns) as a cross-check of the matrix algebra.

In [ ]:
mu_monthly = R.mean().to_numpy()
cov_monthly = R.cov().to_numpy()
rf_mean_monthly = float(rf.mean())

def portfolio_return(w):
  return float(np.dot(w, mu_monthly))

def portfolio_variance(w):
  w = np.asarray(w)
  return float(w @ cov_monthly @ w)

def portfolio_std(w):
  return np.sqrt(portfolio_variance(w))

def portfolio_stats(w, label=""):
  """Expected return, standard deviation and Sharpe ratio (monthly and annualized)."""
  ret_m, sd_m = portfolio_return(w), portfolio_std(w)
  stats = {
    "Monthly expected return": ret_m,
    "Monthly std. deviation": sd_m,
    "Annualized expected return": ret_m * MONTHS_PER_YEAR,
    "Annualized std. deviation": sd_m * np.sqrt(MONTHS_PER_YEAR),
    "Sharpe ratio": (ret_m - rf_mean_monthly) / sd_m,
  }
  return pd.Series(stats, name=label)

w_ew = np.repeat(1 / n, n)
ew_weights = pd.Series(w_ew, index=tickers, name="Weight")
ew_stats = portfolio_stats(w_ew, label="Equal-weighted portfolio")

print("Equal-weight portfolio weights (% of the $100m fund):")
display(ew_weights.round(4).to_frame("Weight"))
print("Equal-weight portfolio statistics:")
display(ew_stats.round(4).to_frame("Value"))

## Task 4 — Minimum-variance frontier and the tangency portfolio

The **minimum-variance frontier** is the set of portfolios offering the lowest variance for each attainable expected return. For a target expected return $\bar{r}$ I solve

$$\min_w \; w^\top \Sigma w \quad \text{subject to} \quad \mathbf{1}^\top w = 1, \qquad \mu^\top w = \bar{r},$$

with long-only bounds $0 \le w_i \le 1$ (set `ALLOW_SHORT = True` in the setup cell to relax this).

The **tangency portfolio** is the efficient portfolio with the highest Sharpe ratio; it is obtained by maximizing the Sharpe ratio over the same feasible set. With a risk-free asset available, the efficient frontier is the straight capital allocation line (CAL) from the risk-free rate through the tangency portfolio.

The cells below define the two optimizations, then sweep target returns across the range of the six assets' means and plot the frontier with the tangency portfolio, the equally weighted portfolio, the individual assets, and the CAL.

In [ ]:
bounds = None if ALLOW_SHORT else [(0, 1) for _ in range(n)]
budget = {"type": "eq", "fun": lambda w: np.sum(w) - 1}

def optimize_min_var(target_return):
  """Minimum-variance weights for a required monthly expected return."""
  result = minimize(
    portfolio_variance,
    x0=np.repeat(1 / n, n),
    method="SLSQP",
    bounds=bounds,
    constraints=[
      budget,
      {"type": "eq", "fun": lambda w: portfolio_return(w) - target_return},
    ],
  )
  if not result.success:
    raise RuntimeError(f"Min-variance optimization failed for target {target_return:.4f}: {result.message}")
  return np.asarray(result.x)

def optimize_tangency():
  """Weights maximizing the monthly Sharpe ratio over the risky assets."""
  result = minimize(
    lambda w: -(portfolio_return(w) - rf_mean_monthly) / portfolio_std(w),
    x0=np.repeat(1 / n, n),
    method="SLSQP",
    bounds=bounds,
    constraints=[budget],
  )
  if not result.success:
    raise RuntimeError(f"Tangency optimization failed: {result.message}")
  return np.asarray(result.x)

In [ ]:
w_tan = optimize_tangency()
w_tan = np.clip(w_tan, 0.0, None)
w_tan = w_tan / w_tan.sum()  # renormalize to remove solver round-off

tan_stats = portfolio_stats(w_tan, label="Tangency portfolio")
tangency_weights = pd.Series(w_tan, index=tickers, name="Weight")

print("Tangency (optimal risky) portfolio weights:")
display(tangency_weights.round(4).to_frame("Weight"))
print("Tangency portfolio statistics:")
display(tan_stats.round(4).to_frame("Value"))

In [ ]:
rows = []
target_grid = np.linspace(mu_monthly.min(), mu_monthly.max(), 41)
for target in target_grid:
  try:
    w = optimize_min_var(target)
  except RuntimeError:
    continue  # infeasible target under the current constraints
  rows.append({
    "Monthly expected return": target,
    "Monthly std. deviation": portfolio_std(w),
  })

frontier = pd.DataFrame(rows)
frontier["Annualized expected return"] = frontier["Monthly expected return"] * MONTHS_PER_YEAR
frontier["Annualized std. deviation"] = frontier["Monthly std. deviation"] * np.sqrt(MONTHS_PER_YEAR)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(frontier["Annualized std. deviation"], frontier["Annualized expected return"], "-o", ms=3, label="Min-variance frontier (long-only)")

# Capital allocation line: from the risk-free rate through the tangency portfolio
cal_sd = np.linspace(0, max(frontier["Annualized std. deviation"]) * 1.05, 50)
cal_ret = ann_rf + (tan_stats["Annualized expected return"] - ann_rf) / tan_stats["Annualized std. deviation"] * cal_sd
ax.plot(cal_sd, cal_ret, "--", color="grey", label="Capital allocation line")
ax.scatter([ann_rf], [ann_rf], marker="o", color="black", zorder=5, label="Risk-free asset (T-bill)")

ax.scatter(tan_stats["Annualized std. deviation"], tan_stats["Annualized expected return"], marker="*", s=250, color="red", zorder=6, label="Tangency portfolio")
ax.scatter(ew_stats["Annualized std. deviation"], ew_stats["Annualized expected return"], marker="D", color="green", zorder=6, label="Equally weighted portfolio")

for i, t in enumerate(tickers):
  ax.scatter(ann_vol.iloc[i], ann_mean.iloc[i], marker="x", color="grey")
  ax.annotate(t, (ann_vol.iloc[i], ann_mean.iloc[i]), textcoords="offset points", xytext=(5, -3), fontsize=8)

ax.set_xlabel("Annualized standard deviation")
ax.set_ylabel("Annualized expected return")
ax.set_title("Minimum-variance frontier, tangency portfolio and capital allocation line")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "frontier.png", dpi=150)
plt.show()

frontier.round(4)

## Task 5 — Is the equally weighted portfolio efficient?

A portfolio is **efficient** only if it lies on the *upper* branch of the minimum-variance frontier. The test: construct the minimum-variance portfolio with **the same expected return as the equally weighted portfolio**. If that optimized portfolio has a strictly lower standard deviation (equivalently, a higher Sharpe ratio), the equal-weight policy is not efficient — and the optimized weights are the efficient replacement delivering the same expected return.

In [ ]:
w_eff = optimize_min_var(ew_stats["Monthly expected return"])
w_eff = np.clip(w_eff, 0.0, None)
w_eff = w_eff / w_eff.sum()

eff_stats = portfolio_stats(w_eff, label="Efficient portfolio (same expected return)")
efficient_weights = pd.Series(w_eff, index=tickers, name="Weight")

print("Efficient portfolio weights (matches the equal-weight expected return):")
display(efficient_weights.round(4).to_frame("Weight"))

comparison = pd.concat([ew_stats, eff_stats], axis=1).T
display(comparison.round(4))

sd_gap = ew_stats["Monthly std. deviation"] - eff_stats["Monthly std. deviation"]
if sd_gap > 1e-4:
  print(f"Conclusion: the equally weighted portfolio is NOT efficient - the efficient "
        f"portfolio with the same expected return cuts the monthly standard deviation "
        f"by {sd_gap:.4%} ({sd_gap / ew_stats['Monthly std. deviation']:.1%} of total risk).")
else:
  print("Conclusion: the equally weighted portfolio is (numerically) efficient for its return level.")

## Task 6 — Efficient allocation meeting the 1% monthly floor return

With a risk-free asset available, every efficient *complete* portfolio is a mix of the **risk-free asset and the tangency portfolio** sitting on the capital allocation line. To hit the client's floor of **1% per month**, I solve for the tangency weight $y$ such that

$$r_f + y\,(r_{\tan} - r_f) = 1\% \quad \Longleftrightarrow \quad y = \frac{1\% - r_f}{r_{\tan} - r_f},$$

the remainder $1 - y$ is invested at the T-bill rate, and the standard deviation scales to $\sigma = y \cdot \sigma_{\tan}$. If $y > 1$ the floor requires borrowing at the T-bill rate (a levered position in the tangency portfolio). Dollar amounts for the $100 million fund are shown alongside the weights.

In [ ]:
floor_monthly = 0.01  # client's floor: 1% per month
FUND_SIZE = 100_000_000

y_tan = (floor_monthly - rf_mean_monthly) / (tan_stats["Monthly expected return"] - rf_mean_monthly)
w_rf_complete = 1 - y_tan
floor_sd_monthly = y_tan * tan_stats["Monthly std. deviation"]

floor_allocation = pd.DataFrame({
  "Weight": [w_rf_complete, y_tan],
  "$ amount": [w_rf_complete * FUND_SIZE, y_tan * FUND_SIZE],
}, index=["Risk-free asset (T-bill)", "Tangency portfolio"])

floor_stats = pd.Series({
  "Monthly expected return": floor_monthly,
  "Monthly std. deviation": floor_sd_monthly,
  "Annualized expected return": floor_monthly * MONTHS_PER_YEAR,
  "Annualized std. deviation": floor_sd_monthly * np.sqrt(MONTHS_PER_YEAR),
})

print("Allocation to achieve the 1% monthly floor:")
display(floor_allocation.round(3))
print("Statistics of the floor portfolio:")
display(floor_stats.round(4).to_frame("Value"))
if y_tan > 1:
  print("Note: weight > 100% in the tangency portfolio means borrowing at the T-bill rate (leverage).")

## Export all results to Excel

Everything needed for the submission is written to `output/EF3320_Assignment1_Challenge.xlsx`, one sheet per task:

- **Monthly Returns** — ticker × month return table (Task 1)
- **Risk-Free Rate** — monthly T-bill rates (Task 1)
- **Asset Statistics** — annualized mean return, standard deviation and Sharpe ratio (Task 2)
- **Equal-Weight Portfolio** — policy portfolio weights and statistics (Task 3)
- **Tangency Portfolio** — weights plus expected return and standard deviation (Task 4)
- **Frontier** — min-variance frontier coordinates used for the Excel chart (Task 4)
- **Efficient vs Equal-Weight** — Task 5 comparison
- **Floor Allocation** — Task 6 allocation and statistics

Key answer rows (asset statistics, tangency/efficient weights, floor allocation) are **bolded** in Excel so the final answers stand out.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

excel_file = OUTPUT_DIR / "EF3320_Assignment1_Challenge.xlsx"

tangency_table = pd.DataFrame({
  "Tangency weight": w_tan,
  "Annualized mean return": [ann_mean[t] for t in tickers],
  "Annualized std. deviation": [ann_vol[t] for t in tickers],
}, index=tickers)

eff_table = pd.concat([ew_weights.rename("Equal weight"), efficient_weights.rename("Efficient weight")], axis=1)
eff_comparison = pd.concat([ew_stats, eff_stats], axis=1)

floor_stats_row = floor_stats.to_frame("Floor portfolio").T

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
  R.to_excel(writer, sheet_name="Monthly Returns")
  rf.to_frame("T-bill monthly rate").to_excel(writer, sheet_name="Risk-Free Rate")
  asset_stats.round(6).to_excel(writer, sheet_name="Asset Statistics")
  ew_weights.round(6).to_frame("Weight").to_excel(writer, sheet_name="Equal-Weight Portfolio")
  ew_stats.round(6).to_frame("Value").to_excel(writer, sheet_name="Equal-Weight Portfolio", startrow=len(tickers) + 3)
  tangency_table.round(6).to_excel(writer, sheet_name="Tangency Portfolio")
  tan_stats.round(6).to_frame("Value").to_excel(writer, sheet_name="Tangency Portfolio", startrow=len(tickers) + 3)
  frontier.round(6).to_excel(writer, sheet_name="Frontier", index=False)
  eff_table.round(6).to_excel(writer, sheet_name="Efficient vs Equal-Weight")
  eff_comparison.round(6).to_excel(writer, sheet_name="Efficient vs Equal-Weight", startrow=len(tickers) + 3)
  floor_allocation.round(6).to_excel(writer, sheet_name="Floor Allocation")
  floor_stats_row.round(6).to_excel(writer, sheet_name="Floor Allocation", startrow=5)

# Bold the key answer rows so they stand out in the submission
wb = load_workbook(excel_file)
bold = Font(bold=True)
key_rows = {
  "Asset Statistics": [1],
  "Equal-Weight Portfolio": [1, len(tickers) + 4],
  "Tangency Portfolio": [1, len(tickers) + 4],
  "Efficient vs Equal-Weight": [1],
  "Floor Allocation": [1, 7],
}
for sheet, rows in key_rows.items():
  ws = wb[sheet]
  for row in rows:
    for cell in ws[row]:
      if cell.value is not None:
        cell.font = bold
wb.save(excel_file)

print(f"Wrote {excel_file.resolve()}")

## Task 7 — Verification with an AI tool

This solution was verified with **opencode** (an AI coding assistant powered by GLM). What I asked it and what it helped with:

1. **Fixing and validating the optimization.** I asked it to check my `scipy` constraint definitions — it spotted that EqualityConstraint dictionaries for `scipy.optimize.minimize` required the `"fun"` key rather than the `"f"` key I had first used, so the tangent-portfolio solve was running unconstrained. It also caught a duplicated CRSP row (NVDA, 2024-06-28) that would have double-counted a month, and it ran the full notebook end-to-end on a test dataset to confirm every statistic and the Excel export before I touched the real data.
2. **Critiquing the portfolio from an investment perspective.** I asked it to assess the tangency portfolio it produced. Its critique: the optimal long-only weights concentrate very heavily in a couple of high-Sharpe assets, so the result is highly sensitive to estimation error — five years of monthly data is short, and 2020–2024 is a very unusual sample (COVID crash, 2022 bond-equity selloff, AI-driven tech rally). As a fund manager I would therefore treat the tangency weights as a tilt rather than a literal allocation, consider shrinkage or resampling of the covariance matrix, and stress-test the floor-return allocation under leverage with the borrowing assumption noted.










